# Gradio for creating User Interface for LLM models

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown
import gradio as gr

In [4]:
load_dotenv(override = True)
open_ai_key = os.getenv("OPENAI_API_KEY")

if open_ai_key:
    print("Key is loaded!")
else:
    print("Not loaded!!")

openai = OpenAI()

Key is loaded!


In [ ]:
# system_message = "You are a really sweet and a very helpful assistant"

# def generate_response(prompt):
#     messages = [{"role":"system","content":"system_message"},{"role":"user","content":prompt}]
#     response = openai.chat.completions.create(
#         model = "gpt-5-nano",
#         messages = messages
#     )
#     return display(Markdown(response.choices[0].message.content))

In [ ]:
# user_message = "What is today's date?"

# generate_response(user_message)

Today is October 7, 2026. Would you like it in another format?

## Creating a simple Gradio Interface:

In [10]:
def example(animal):
    print(f"My favorite animal is {animal}")
    return f"My favorite animal is {animal.upper()}"

In [ ]:
"""the launch function takes in a parameter called "share" which takes either True or False, which basically hosts your gradio app on hugging face allowing anyone to interact with it. 
Though this is mostly used in data science or other shareable projects. Though the caveat is that, if others try running your app, hugging face runs the code on you system first which 
makes thing a bit risky, if you code contains sensitive data.
"""
gr.Interface(fn = example, inputs = "textbox", outputs = "textbox", flagging_mode = "never").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


My favorite animal is cat


# Adding authentication

In [12]:
"""
"inbrowser" => forces gradio to launch in browser.
"auth" => Makes authentication available for the gradio interface.
"""

gr.Interface(fn = example, inputs = "textbox", outputs = "textbox", flagging_mode = "never").launch(inbrowser = True, auth = ("dark", "11223344"))

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


My favorite animal is Dog


## Force dark mode:

In [ ]:
force_dark_mode = """
function refresh(){
    const url = new URL(window.location);
    if (url.searchParams.get('__theme') !== 'dark'){ 
        url.searchParams.set('__theme'), 'dark');
        window.location.href = url.href;

    }
}
"""
# swapping 'dark' to 'light' enables light mode in 'force_dark_mode'

gr.Interface(fn = example, inputs = "textbox", outputs = "textbox", flagging_mode = "never", js = force_dark_mode).launch()

## Accepting user prompt from user through gradio TextBox

In [ ]:
message_input = gr.Textbox(label = "Your message", info = "Enter a your favorite animal!", lines=7)
message_output = gr.Textbox(label = "Response", lines=8)

gr.Interface(
    fn = example,
    title = "Animals",
    inputs = message_input,
    outputs = message_output,
    examples = ["Goat","Otter"],
    flagging_mode = "never"
).launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


My favorite animal is Dog
My favorite animal is Otter


## Writing a LLM interface:

In [26]:
system_message = "You are a really sweet and a very helpful assistant who responds in Markdown without code blocks"

def generate_response(prompt):
    messages = [{"role":"system","content":"system_message"},{"role":"user","content":prompt}]
    response = openai.chat.completions.create(
        model = "gpt-5-nano",
        messages = messages
    )
    return response.choices[0].message.content

In [29]:
user_message = gr.Textbox(label = "user_message", info = "Enter prompt", lines=7)
llm_output = gr.Markdown(label = "response")

gr.Interface(
    fn = generate_response,
    title = "LLM Interface",
    inputs = user_message,
    outputs = llm_output,
    flagging_mode = "never"
).launch()


* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


## Gradio Interface but with streaming output.

In [ ]:
def streaming(prompt):
    messages = [{"role":"system","content":system_message},{"role":"user","content":prompt}]
    stream = openai.chat.completions.create(
        model = "gpt-5-nano",
        messages = messages,
        stream = True
    )

    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [38]:
input_prompt = gr.Textbox(label = "user prompt", info = "Enter prompt", lines = 7)
output_message = gr.Markdown(label = "output response")

gr.Interface(
    fn = streaming,
    title = "Streaming LLM Interface",
    inputs = input_prompt,
    outputs = output_message,
    flagging_mode = "never"
).launch()

* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


## Multi-GPT-Model Gradio Interface:

In [39]:
def streaming(prompt,model):
    messages = [{"role":"system","content":system_message},{"role":"user","content":prompt}]
    stream = openai.chat.completions.create(
        model = model,
        messages = messages,
        stream = True
    )

    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [42]:
input_prompt = gr.Textbox(label = "user prompt", info = "Enter prompt", lines = 7)
model_selector = gr.Dropdown(["gpt-5-nano","gpt-4.1-nano","gpt-5"], label = "Model_selector",value = "gpt-5-nano")
output_message = gr.Markdown(label = "output response")

gr.Interface(
    fn = streaming,
    title = "Streaming LLM Interface",
    inputs = [input_prompt, model_selector],
    outputs = output_message,
    flagging_mode = "never"
).launch()

* Running on local URL:  http://127.0.0.1:7875
* To create a public link, set `share=True` in `launch()`.


## Company Brochure generator:

In [43]:
brochure_system_prompt = """
    You are a marketing expert. Based on the given webiste contents, relevant links and the contents of those links, create
    a brochure for the website. The brochure should be structured and should include the following details:
    1. Website Overview: A brief introduction to the website, its purpose, and its target audience.
    2. Key Features: Highlight the main features of the website, explaining how they benefit users.
    3. Use Cases: Provide examples of how different users can utilize the product or service offered by the website.
    4. Careers: If applicable, include information about career opportunities, company culture, and how to apply.
    5. Documentation: Summarize the available documentation, guides, or resources that help users understand and use the website effectively.
    The brochure should be concise, engaging, and informative, making it easy for readers to understand the value of the website and its offerings."""

In [45]:
def streaming_bro(prompt,model):
    messages = [{"role":"system","content":brochure_system_prompt},{"role":"user","content":prompt}]
    stream = openai.chat.completions.create(
        model = model,
        messages = messages,
        stream = True
    )

    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [55]:
from scraping import fetch_website_contents

def stream_brochure(compnay_name, url, model):
    yield ""
    prompt = f"Please generate a company brochure for {company_name}. Here is there landing page:\n"
    str_url = str(url)
    prompt+=fetch_website_contents(str_url)
    result = streaming_bro(prompt, model)

    yield from result

In [56]:
company_name = gr.Textbox(label = "company_name", info = "Enter company name", lines = 7)
url_line = gr.Textbox(label = "url_string", info = "Enter the url", lines = 7)
model_selector = gr.Dropdown(["gpt-5-nano","gpt-4.1-nano","gpt-5"], label = "Model_selector",value = "gpt-5-nano")
output_message = gr.Markdown(label = "output response")

gr.Interface(
    fn = stream_brochure,
    title = "Streaming LLM Interface",
    inputs = [company_name, url_line, model_selector],
    outputs = output_message,
    flagging_mode = "never"
).launch()

* Running on local URL:  http://127.0.0.1:7881
* To create a public link, set `share=True` in `launch()`.
